Stacking CatBoost, LightGBM, XGB jsut for fun

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score
from sklearn.base import clone

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from feature_generators.temporal_features import TemporalFeatures
from feature_generators.behavioral_features import BehavioralFeatures
from feature_generators.gps_features import GPSFeatures


In [11]:
outcomes = pd.read_csv('dataset/loan_outcomes_train.csv')
features = pd.read_csv('dataset/features.csv')
events = pd.read_csv('dataset/events.csv')
gps = pd.read_csv('dataset/gps.csv')

In [12]:
# 1. temporal featrues
temporal_builder = TemporalFeatures(outcomes_df=outcomes)
features_merged = temporal_builder.add_temporal_features()

# 2. GPS features
gps_builder = GPSFeatures(gps_df=gps, outcomes_df=features_merged)
features_merged = gps_builder.add_gps_features()

# 3. behavior features
behavioral_builder = BehavioralFeatures(
    events_df=events, 
    outcomes_df=features_merged,
    window_days=14
)
features_merged = behavioral_builder.add_behavioral_features(
    include_bigrams=True,
    include_ranks=True,
    include_diligence_stability=True,
    drop_raw=True
)

# merge with the given masked features
features_merged = features_merged.merge(
    features,
    on='user_id',
    how='inner'
)

# apparently missingness from 3 is a valualble feature so adding that, refer feature_importance notebook
features_merged['feature_3_is_missing'] = features_merged['feature_3'].isna().astype(int)

print(f"Final shape: {features_merged.shape}")
print(f"Columns: {list(features_merged.columns)}")

c:\Users\soham\OneDrive\Desktop\main\loan_outcome_prediction\gps_features.py:54: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x[['lat_round', 'lon_round']].drop_duplicates().shape[0])
c:\Users\soham\OneDrive\Desktop\main\loan_outcome_prediction\gps_features.py:105: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(rog)
c:\Users\soham\OneDrive\Desktop\main\loan_outcome_prediction\behavioral_featur

Final shape: (8089, 44)
Columns: ['user_id', 'application_at', 'is_repaid', 'app_hour', 'app_day_of_week', 'app_day_of_month', 'app_year', 'is_weekend', 'is_late_night', 'max_land_speed', 'gps_points_pre_app', 'unique_locations', 'dominant_location_ratio', 'location_entropy', 'radius_of_gyration', 'gps_points_pre_app_log', 'radius_of_gyration_log', 'num_events_14d', 'count_ph_inst', 'num_sessions_rank', 'active_days_rank', 'avg_session_duration', 'pct_single_event_sessions', 'events_last_24h', 'time_from_last_event_to_application', 'num_unique_screens', 'events_last_24h_ratio', 'has_events_14d', 'session_duration_rank', 'recency_rank', 'diligence_stability', 'mean_bigram_default_rate', 'max_bigram_default_rate', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_3_is_missing']


In [13]:
# chronological sorting to prevent data leakage
features_merged = features_merged.sort_values('application_at').reset_index(drop=True)

# split 90/10
split_idx = int(len(features_merged) * 0.9)
train_df = features_merged.iloc[:split_idx]
val_df = features_merged.iloc[split_idx:]

In [14]:
X_cols = [
    # Masked user features (Features)
    'feature_1', 
    'feature_2',
    'feature_4', 
    'feature_8', 
    'feature_9',
    'feature_10',
    
    # Missingness as a feature (Features)
    'feature_3_is_missing',
    
    # GPS & Location features (GPS)
    'count_ph_inst',
    'gps_points_pre_app',
    'unique_locations',
    'dominant_location_ratio',
    'location_entropy',
    'max_land_speed',
    'radius_of_gyration_log',
    'gps_points_pre_app_log',

    # Temporal features (Temporal)
    'app_hour',
    'app_day_of_week',
    'app_day_of_month',
    'app_year',
    'is_weekend',
    'is_late_night',

    # Behavioral ranks & interactions (Events)
    'num_sessions_rank',
    'active_days_rank',
    'session_duration_rank',
    'recency_rank',
    'diligence_stability',
    # if i remove these 2 features, all metrics get better for all models except xgb (leakage?)
    'mean_bigram_default_rate',
    'max_bigram_default_rate',
]


X_train = train_df[X_cols]
y_train = train_df['is_repaid']

X_val = val_df[X_cols]
y_val = val_df['is_repaid']

print(X_train.shape, y_train.shape)

(7280, 28) (7280,)


In [15]:

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    verbose=False
)

lgb_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8
)

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    use_label_encoder=False
)

base_models = {
    "cat": cat_model,
    "lgb": lgb_model,
    "xgb": xgb_model
}



## Out-of-Fold Predictions (No Leakage)


In [ ]:
from sklearn.base import clone

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_preds = {name: np.zeros(len(X_train)) for name in base_models}

fitted_models = {name: [] for name in base_models}

for name, model in base_models.items():
    print(f"Training {name}...")
    for fold_idx, (train_idx, oof_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr_fold, X_oof_fold = X_train.iloc[train_idx], X_train.iloc[oof_idx]
        y_tr_fold = y_train.iloc[train_idx]

        fold_model = clone(model)
        fold_model.fit(X_tr_fold, y_tr_fold)
        
        oof_preds[name][oof_idx] = fold_model.predict_proba(X_oof_fold)[:, 1]
        fitted_models[name].append(fold_model)

stack_train = np.column_stack([oof_preds[m] for m in base_models])

print("\nOOF AUC scores:")
for name in base_models:
    oof_auc = roc_auc_score(y_train, oof_preds[name])
    print(f"  {name}: {oof_auc:.4f}")


Training cat...
Training lgb...
[LightGBM] [Info] Number of positive: 1973, number of negative: 3851
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000729 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3339
[LightGBM] [Info] Number of data points in the train set: 5824, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.338771 -> initscore=-0.668778
[LightGBM] [Info] Start training from score -0.668778
[LightGBM] [Info] Number of positive: 1973, number of negative: 3851
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000580 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3330
[LightGBM] [Info] Number of data points in the train set: 5824, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.338771 -> initscore=-0.668778
[LightGBM] [Info] Start training from sc

c:\Users\soham\OneDrive\Desktop\main\loan_outcome_prediction\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [23:33:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\soham\OneDrive\Desktop\main\loan_outcome_prediction\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [23:33:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\soham\OneDrive\Desktop\main\loan_outcome_prediction\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [23:33:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\soham\OneDrive\Desktop\main\loan_outcome_prediction\.venv\Lib\site-packages\xgboost\training.p


OOF AUC scores (on training set):
  cat: 0.6082
  lgb: 0.6034
  xgb: 0.6053


In [17]:

meta_models = {
    "logreg": LogisticRegression(max_iter=1000),
    "logreg_l2": LogisticRegression(max_iter=1000, penalty="l2", C=0.5)
}

results = []

for name, meta in meta_models.items():
    meta.fit(stack_train, y_train)
    preds = meta.predict_proba(stack_train)[:, 1]

    results.append({
        "meta_model": name,
        "auc": roc_auc_score(y_train, preds),
        "logloss": log_loss(y_train, preds),
        "accuracy": accuracy_score(y_train, (preds > 0.5).astype(int))
    })

pd.DataFrame(results)


c:\Users\soham\OneDrive\Desktop\main\loan_outcome_prediction\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,meta_model,auc,logloss,accuracy
0,logreg,0.612738,0.623002,0.664835
1,logreg_l2,0.612761,0.623026,0.665797


In [ ]:
final_base_models = {}
for name, model in base_models.items():
    print(f"Refitting {name} on full training set...")
    final_model = clone(model)
    final_model.fit(X_train, y_train)
    final_base_models[name] = final_model

stack_valid = np.column_stack([
    final_base_models[name].predict_proba(X_val)[:, 1] for name in base_models
])

best_meta = LogisticRegression(max_iter=1000)
best_meta.fit(stack_train, y_train)

final_preds = best_meta.predict_proba(stack_valid)[:, 1]

print("\n" + "="*50)
print("STACKED ENSEMBLE METRICS (10% HOLDOUT)")
print("="*50)
print(f"AUC:      {roc_auc_score(y_val, final_preds):.4f}")
print(f"LogLoss:  {log_loss(y_val, final_preds):.4f}")
print(f"Accuracy: {accuracy_score(y_val, (final_preds > 0.5).astype(int)):.4f}")

print("\n" + "="*50)
print("INDIVIDUAL BASE MODEL METRICS (10% HOLDOUT)")
print("="*50)
for name, model in final_base_models.items():
    preds = model.predict_proba(X_val)[:, 1]
    print(f"{name:>6} - AUC: {roc_auc_score(y_val, preds):.4f}, LogLoss: {log_loss(y_val, preds):.4f}")


Refitting cat on full training set...
Refitting lgb on full training set...
[LightGBM] [Info] Number of positive: 2467, number of negative: 4813
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000441 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3385
[LightGBM] [Info] Number of data points in the train set: 7280, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.338874 -> initscore=-0.668318
[LightGBM] [Info] Start training from score -0.668318
Refitting xgb on full training set...


c:\Users\soham\OneDrive\Desktop\main\loan_outcome_prediction\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [23:34:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



STACKED ENSEMBLE METRICS (10% HOLDOUT)
AUC:      0.5826
LogLoss:  0.6095
Accuracy: 0.6922

INDIVIDUAL BASE MODEL METRICS (10% HOLDOUT)
   cat - AUC: 0.5854, LogLoss: 0.6143
   lgb - AUC: 0.5737, LogLoss: 0.6359
   xgb - AUC: 0.5677, LogLoss: 0.6421
